In [65]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor


In [66]:
data = pd.read_csv('e1_nutrients.csv')

In [ ]:
nutrient_cols = [c for c in data.columns if c != "Depth"]
plt.figure(figsize=(10, 6))
for col in nutrient_cols:
    plt.scatter(data["Depth"], data[col], s=35, alpha=0.75, label=col)

plt.xlabel("Depth")
plt.ylabel("Concentration")
plt.title("Scatter Plot of All Nutrient Data vs Depth")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def filter_by_column(data, column_name):
    if column_name in data.columns:
        if "Depth" not in data.columns:
            raise ValueError("Column 'Depth' does not exist in the Data.")
        
        data_new = data.copy()

        data_new = data_new.sort_values("Depth", kind="mergesort").reset_index(drop=True)

        q1 = data_new.groupby("Depth")[column_name].quantile(0.25)
        q3 = data_new.groupby("Depth")[column_name].quantile(0.75)

        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        lower_mapped = data_new["Depth"].map(lower_bound)
        upper_mapped = data_new["Depth"].map(upper_bound)

        filtered_data = data_new[
            (data_new[column_name] >= lower_mapped) & 
            (data_new[column_name] <= upper_mapped)
        ]
        
        filtered_data = filtered_data.reset_index(drop=True)
        
        return filtered_data
    
    else:
        raise ValueError(f"Column '{column_name}' does not exist in the Data.")

columns_to_clean = ['NITRITE', 'NITRATE+NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE']
clean_data = data.copy()
for col in columns_to_clean:
    clean_data = filter_by_column(clean_data, col)

print(clean_data)
